In [1]:
import sys
print(f"Đường dẫn Python: {sys.executable}")
import os
print(f"Thư mục hiện tại: {os.getcwd()}")

Đường dẫn Python: /home/shibe/python_envs/cdc_project/bin/python
Thư mục hiện tại: /mnt/d/Data_engineer/Data app/recruitment_cdc_project


In [2]:
# Cài đặt các thư viện còn thiếu cho file tạo data
!pip install pandas mysql-connector-python sqlalchemy faker

In [3]:
import sys
!{sys.executable} -m pip install cassandra-driver pandas mysql-connector-python faker sqlalchemy

In [5]:
from cassandra.cluster import Cluster
from cassandra.cqlengine import columns
from cassandra.cqlengine.models import Model
from cassandra.cqlengine.management import sync_table
from cassandra.cqlengine import connection
from cassandra.query import dict_factory
from datetime import datetime, timedelta
import time
import cassandra
import random
import uuid
import math
import pandas as pd
import datetime
from sqlalchemy import create_engine
import numpy as np
import mysql.connector
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

# Thiết lập hiển thị Pandas
pd.set_option("display.max_rows", None, "display.max_columns", None)

# --- Cấu hình toàn cục (Giữ nguyên tên biến) ---
host = 'localhost'
port = '3306'
db_name = 'schema_name' # Đã cập nhật theo ảnh schema_name của bạn
user = 'root'
password = 'root'
# URL cho SQLAlchemy
db_url = f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{db_name}"
engine = create_engine(db_url)

keyspace = 'keyspace_name' # Đã cập nhật theo ảnh keyspace_name của bạn
cassandra_login = 'cassandra'
cassandra_password = 'cassandra'
contact_points = ['127.0.0.1']

# Cassandra mặc định của Docker image 3.11 thường có user/pass là cassandra/cassandra
auth_provider = PlainTextAuthProvider(username='cassandra', password='cassandra')

cluster = Cluster(contact_points, port=9042, auth_provider=auth_provider)

# Thử kết nối và tạo session 
session = cluster.connect() # Kết nối trước để kiểm tra
session.set_keyspace(keyspace) # Chọn keyspace đã tạo

def get_data_from_job(user, password, host, database):
    # Sử dụng SQLAlchemy engine để tránh UserWarning
    query = "SELECT id AS job_id, campaign_id, group_id, company_id FROM job"
    mysql_data = pd.read_sql(query, engine)
    return mysql_data

def get_data_from_publisher(user, password, host, database):
    # Hiệu chỉnh lấy id từ bảng company làm publisher_id
    query = "SELECT id AS publisher_id FROM company"
    mysql_data = pd.read_sql(query, engine)
    return mysql_data

def generating_dummy_data(n_records, session, user, password, host, db_name):
    # Giữ nguyên phần lấy list dữ liệu cực kỳ trực quan của shibe
    publisher = get_data_from_publisher(user, password, host, db_name)
    publisher = publisher['publisher_id'].to_list()
    jobs_data = get_data_from_job(user, password, host, db_name)
    job_list = jobs_data['job_id'].to_list()
    campaign_list = jobs_data['campaign_id'].to_list()
    group_list = jobs_data[jobs_data['group_id'].notnull()]['group_id'].astype(int).to_list()
    
    i = 0 
    fake_records = n_records
    while i <= fake_records:
        # MẸO: Giữ nguyên là đối tượng UUID của Cassandra, dùng .format tự nó sẽ ép kiểu
        create_time = cassandra.util.uuid_from_time(datetime.datetime.now())
        
        bid = random.randint(0, 1)
        interact = ['click', 'conversion', 'qualified', 'unqualified']
        custom_track = random.choices(interact, weights=(70, 10, 10, 10))[0]
        
        job_id = random.choice(job_list)
        publisher_id = random.choice(publisher)
        group_id = random.choice(group_list)
        campaign_id = random.choice(campaign_list)
        ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        # CHÚ Ý: Bỏ nháy đơn ở {} đầu tiên của create_time
        # Cassandra nhận diện TimeUUID không nháy đơn là hằng số (constant)
        sql = """ 
            INSERT INTO tracking (create_time, bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts) 
            VALUES ({}, {}, {}, '{}', {}, {}, {}, '{}')
        """.format(create_time, bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts)
        
        print(sql)
        session.execute(sql)
        i += 1 
        
    return print("Data Generated Successfully")
# --- Vòng lặp thực thi (Giữ nguyên) ---
status = "ON"
while status == "ON":
    generating_dummy_data(n_records=random.randint(1, 20), session=session, user=user, password=password, host=host, db_name=db_name)
    time.sleep(20)

 
            INSERT INTO tracking (create_time, bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts) 
            VALUES (8311b9b6-2c54-11f1-a4f7-026c3decb2ff, 1, 1, 'qualified', 22, 1434, 51, '2026-03-30 16:21:31')
        
 
            INSERT INTO tracking (create_time, bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts) 
            VALUES (8314370e-2c54-11f1-9e98-c54c608e3907, 1, 57, 'qualified', 32, 661, 163, '2026-03-30 16:21:31')
        
Data Generated Successfully
 
            INSERT INTO tracking (create_time, bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts) 
            VALUES (8f049db0-2c54-11f1-9778-5fba4ab1ce38, 1, 53, 'click', 22, 1960, 182, '2026-03-30 16:21:51')
        
 
            INSERT INTO tracking (create_time, bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts) 
            VALUES (8f056f88-2c54-11f1-8154-0a42744036d4, 0, 178, 'click', 41, 93, 185, '2026-03-30 16:21:51')
        
 
          

KeyboardInterrupt: 